# N3: Stacking for Model 5 (n0 + n1)

Build Model 5 n0 and n1 with 2 stacking approaches each:
1. **Approach 1**: LogReg meta-learner on 2 outputs only
2. **Approach 2**: LightGBM meta-learner on input features + 2 outputs

Compare both approaches with individual sub-models.

## 1. Import Libraries & Device Setup

In [1]:
import numpy as np
import pandas as pd
import warnings
import torch
import gc
from tqdm import tqdm
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from transformers import AutoTokenizer, AutoModel

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("✅ All imports successful")
print(f"   Device: {device}")

✅ All imports successful
   Device: cuda


## 2. Load Data from CSV

In [2]:
train_path = '../../../data/splited/train_set.csv'
val_path = '../../../data/splited/val_set.csv'

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

y_train = df_train['label'].values
y_val = df_val['label'].values

print(f"Data loaded: Train {df_train.shape} | Val {df_val.shape}")
print(f"Labels: Train {(y_train==0).sum()} fake / {(y_train==1).sum()} real")
print(f"        Val {(y_val==0).sum()} fake / {(y_val==1).sum()} real")

Data loaded: Train (3788, 28) | Val (474, 28)
Labels: Train 3143 fake / 645 real
        Val 393 fake / 81 real


## 3. Generate TF-IDF Embeddings

In [3]:
texts_train_strict = df_train['text_strict'].fillna('').tolist()
texts_val_strict = df_val['text_strict'].fillna('').tolist()

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf_full = tfidf.fit_transform(texts_train_strict)
X_val_tfidf_full = tfidf.transform(texts_val_strict)

svd = TruncatedSVD(n_components=900, random_state=42)
X_train_tfidf = svd.fit_transform(X_train_tfidf_full)
X_val_tfidf = svd.transform(X_val_tfidf_full)

print(f"✅ TF-IDF: Train {X_train_tfidf.shape} | Val {X_val_tfidf.shape}")

✅ TF-IDF: Train (3788, 900) | Val (474, 900)


## 4. Extract PhoBERT Embeddings

In [4]:
texts_train_loose = df_train['text_loose'].fillna('').tolist()
texts_val_loose = df_val['text_loose'].fillna('').tolist()

def extract_phobert_embeddings(texts, batch_size=16):
    tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
    model = AutoModel.from_pretrained('vinai/phobert-base-v2').to(device).eval()
    embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="PhoBERT", leave=False):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs, output_hidden_states=True)
            cls_tokens = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(cls_tokens)
            del outputs, inputs
            torch.cuda.empty_cache()
    
    model.to('cpu')
    del model, tokenizer
    gc.collect()
    return np.array(embeddings)

X_train_phobert = extract_phobert_embeddings(texts_train_loose, batch_size=16)
X_val_phobert = extract_phobert_embeddings(texts_val_loose, batch_size=16)

print(f"✅ PhoBERT: Train {X_train_phobert.shape} | Val {X_val_phobert.shape}")

Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ PhoBERT: Train (3788, 768) | Val (474, 768)


## 5. Extract Hand-Crafted Numeric Features

In [ ]:
exclude_cols = {'id', 'label', 'text_strict', 'text_loose', 'post_message'}
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in numeric_cols 
                if col not in exclude_cols and col.strip() != '' and not col.startswith('Unnamed')]

if feature_cols:
    X_train_features = df_train[feature_cols].fillna(0).values
    X_val_features = df_val[feature_cols].fillna(0).values
else:
    X_train_features = np.array([]).reshape(len(df_train), 0)
    X_val_features = np.array([]).reshape(len(df_val), 0)

print(f"✅ Hand-crafted features: {len(feature_cols)} features")
print(f"   Train {X_train_features.shape} | Val {X_val_features.shape}")

✅ Hand-crafted features: 23 features
   Train (3788, 23) | Val (474, 23)


## 6. Combine All Features & Scale

In [ ]:
X_train_combined = np.hstack([X_train_tfidf, X_train_phobert, X_train_features])
X_val_combined = np.hstack([X_val_tfidf, X_val_phobert, X_val_features])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_combined)
X_val_scaled = scaler.transform(X_val_combined)

total_dims = X_train_tfidf.shape[1] + X_train_phobert.shape[1] + X_train_features.shape[1]
print(f"✅ Combined features: {total_dims} dimensions")
print(f"   TF-IDF: {X_train_tfidf.shape[1]} + PhoBERT: {X_train_phobert.shape[1]} + Hand-crafted: {X_train_features.shape[1]}")
print(f"   Train {X_train_scaled.shape} | Val {X_val_scaled.shape}")

✅ Combined features: 1691 dimensions
   TF-IDF: 900 + PhoBERT: 768 + Hand-crafted: 23
   Train (3788, 1691) | Val (474, 1691)


# MODEL 5 n0: LogReg + RandomForest

## 7. Build Model 5 n0 Sub-models (5a + 5b)

In [7]:
print("\n" + "="*80)
print("MODEL 5 n0: LogisticRegression (5a) + RandomForest (5b)")
print("="*80)

# Sub-model 5a: LogisticRegression
print("\nTraining 5a: LogisticRegression...")
model5_n0_5a = LogisticRegression(
    penalty='l2', C=1.0, max_iter=2000, solver='lbfgs',
    class_weight='balanced', random_state=42, verbose=0
)
model5_n0_5a.fit(X_train_scaled, y_train)
print("✅ Model 5a trained")

# Sub-model 5b: RandomForest
print("\nTraining 5b: RandomForest...")
model5_n0_5b = RandomForestClassifier(
    n_estimators=100, max_depth=15, min_samples_split=5,
    min_samples_leaf=2, class_weight='balanced', random_state=42, n_jobs=-1, verbose=0
)
model5_n0_5b.fit(X_train_scaled, y_train)
print("✅ Model 5b trained")

# Get predictions for validation
print("\nGenerating predictions...")
y_val_pred_5a_n0 = model5_n0_5a.predict(X_val_scaled)
y_val_proba_5a_n0 = model5_n0_5a.predict_proba(X_val_scaled)[:, 1]

y_val_pred_5b_n0 = model5_n0_5b.predict(X_val_scaled)
y_val_proba_5b_n0 = model5_n0_5b.predict_proba(X_val_scaled)[:, 1]

print("✅ Predictions generated")


MODEL 5 n0: LogisticRegression (5a) + RandomForest (5b)

Training 5a: LogisticRegression...
✅ Model 5a trained

Training 5b: RandomForest...
✅ Model 5b trained

Generating predictions...
✅ Predictions generated


## 8. Stacking Approach 1 n0: LogReg on Outputs Only

In [8]:
print("\n" + "="*80)
print("APPROACH 1 n0: LogReg Meta-learner on 2 Outputs")
print("="*80)

# Get train predictions for meta-features
y_train_proba_5a_n0 = model5_n0_5a.predict_proba(X_train_scaled)[:, 1]
y_train_proba_5b_n0 = model5_n0_5b.predict_proba(X_train_scaled)[:, 1]

# Create meta-features (outputs only)
X_train_meta_n0_app1 = np.column_stack([y_train_proba_5a_n0, y_train_proba_5b_n0])
X_val_meta_n0_app1 = np.column_stack([y_val_proba_5a_n0, y_val_proba_5b_n0])

print(f"Meta-features shape: Train {X_train_meta_n0_app1.shape} | Val {X_val_meta_n0_app1.shape}")

# Train meta-learner (LogReg)
print("\nTraining LogReg meta-learner...")
stacking_n0_app1 = LogisticRegression(
    penalty='l2', C=1.0, max_iter=2000, solver='lbfgs',
    class_weight='balanced', random_state=42, verbose=0
)
stacking_n0_app1.fit(X_train_meta_n0_app1, y_train)
print("✅ Meta-learner trained")

# Predictions
y_val_pred_stack_n0_app1 = stacking_n0_app1.predict(X_val_meta_n0_app1)
y_val_proba_stack_n0_app1 = stacking_n0_app1.predict_proba(X_val_meta_n0_app1)[:, 1]

# Metrics
f1_stack_n0_app1 = f1_score(y_val, y_val_pred_stack_n0_app1, average='weighted')
auc_stack_n0_app1 = roc_auc_score(y_val, y_val_proba_stack_n0_app1)
acc_stack_n0_app1 = accuracy_score(y_val, y_val_pred_stack_n0_app1)
prec_stack_n0_app1 = precision_score(y_val, y_val_pred_stack_n0_app1, average='weighted')
rec_stack_n0_app1 = recall_score(y_val, y_val_pred_stack_n0_app1, average='weighted')

print(f"\n✅ Approach 1 Results:")
print(f"   F1: {f1_stack_n0_app1:.4f} | AUC: {auc_stack_n0_app1:.4f} | Acc: {acc_stack_n0_app1:.4f}")
print(f"   Prec: {prec_stack_n0_app1:.4f} | Rec: {rec_stack_n0_app1:.4f}")


APPROACH 1 n0: LogReg Meta-learner on 2 Outputs
Meta-features shape: Train (3788, 2) | Val (474, 2)

Training LogReg meta-learner...
✅ Meta-learner trained

✅ Approach 1 Results:
   F1: 0.9086 | AUC: 0.9503 | Acc: 0.9093
   Prec: 0.9080 | Rec: 0.9093


## 9. Stacking Approach 2 n0: LightGBM on Inputs + Outputs

In [9]:
print("\n" + "="*80)
print("APPROACH 2 n0: LightGBM Meta-learner on Input Features + 2 Outputs")
print("="*80)

# Create meta-features (inputs + outputs)
X_train_meta_n0_app2 = np.column_stack([
    y_train_proba_5a_n0, y_train_proba_5b_n0,
    X_train_scaled
])
X_val_meta_n0_app2 = np.column_stack([
    y_val_proba_5a_n0, y_val_proba_5b_n0,
    X_val_scaled
])

print(f"Meta-features shape: Train {X_train_meta_n0_app2.shape} | Val {X_val_meta_n0_app2.shape}")

# Train meta-learner (LightGBM)
print("\nTraining LightGBM meta-learner...")
stacking_n0_app2 = LGBMClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=7, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, is_unbalance=True,
    random_state=42, verbose=-1
)
stacking_n0_app2.fit(X_train_meta_n0_app2, y_train)
print("✅ Meta-learner trained")

# Predictions
y_val_pred_stack_n0_app2 = stacking_n0_app2.predict(X_val_meta_n0_app2)
y_val_proba_stack_n0_app2 = stacking_n0_app2.predict_proba(X_val_meta_n0_app2)[:, 1]

# Metrics
f1_stack_n0_app2 = f1_score(y_val, y_val_pred_stack_n0_app2, average='weighted')
auc_stack_n0_app2 = roc_auc_score(y_val, y_val_proba_stack_n0_app2)
acc_stack_n0_app2 = accuracy_score(y_val, y_val_pred_stack_n0_app2)
prec_stack_n0_app2 = precision_score(y_val, y_val_pred_stack_n0_app2, average='weighted')
rec_stack_n0_app2 = recall_score(y_val, y_val_pred_stack_n0_app2, average='weighted')

print(f"\n✅ Approach 2 Results:")
print(f"   F1: {f1_stack_n0_app2:.4f} | AUC: {auc_stack_n0_app2:.4f} | Acc: {acc_stack_n0_app2:.4f}")
print(f"   Prec: {prec_stack_n0_app2:.4f} | Rec: {rec_stack_n0_app2:.4f}")


APPROACH 2 n0: LightGBM Meta-learner on Input Features + 2 Outputs
Meta-features shape: Train (3788, 1693) | Val (474, 1693)

Training LightGBM meta-learner...
✅ Meta-learner trained

✅ Approach 2 Results:
   F1: 0.9101 | AUC: 0.9417 | Acc: 0.9156
   Prec: 0.9120 | Rec: 0.9156


## 10. Comparison n0: Ensemble vs 2 Stacking

In [20]:
print("\n" + "="*80)
print("COMPARISON n0: Ensemble vs 2 Stacking Approaches")
print("="*80)

# Calculate Ensemble (average of 5a + 5b)
y_val_proba_ensemble_n0 = (y_val_proba_5a_n0 + y_val_proba_5b_n0) / 2
y_val_pred_ensemble_n0 = (y_val_proba_ensemble_n0 >= 0.5).astype(int)

# Metrics for Ensemble
f1_ensemble_n0 = f1_score(y_val, y_val_pred_ensemble_n0, average='weighted')
auc_ensemble_n0 = roc_auc_score(y_val, y_val_proba_ensemble_n0)
acc_ensemble_n0 = accuracy_score(y_val, y_val_pred_ensemble_n0)
prec_ensemble_n0 = precision_score(y_val, y_val_pred_ensemble_n0, average='weighted')
rec_ensemble_n0 = recall_score(y_val, y_val_pred_ensemble_n0, average='weighted')

results_n0 = pd.DataFrame({
    'Model': ['Ensemble (5a+5b)/2', 'Stacking Approach 1 (LogReg)', 'Stacking Approach 2 (LightGBM)'],
    'F1': [f1_ensemble_n0, f1_stack_n0_app1, f1_stack_n0_app2],
    'AUC': [auc_ensemble_n0, auc_stack_n0_app1, auc_stack_n0_app2],
    'Accuracy': [acc_ensemble_n0, acc_stack_n0_app1, acc_stack_n0_app2],
    'Precision': [prec_ensemble_n0, prec_stack_n0_app1, prec_stack_n0_app2],
    'Recall': [rec_ensemble_n0, rec_stack_n0_app1, rec_stack_n0_app2]
})

print("\n" + results_n0.to_string(index=False))
best_idx_n0 = results_n0['F1'].idxmax()
print(f"\n🏆 Best n0: {results_n0.loc[best_idx_n0, 'Model']} (F1: {results_n0.loc[best_idx_n0, 'F1']:.4f})")

# Complexity
print(f"\nCOMPLEXITY:")
print(f"  Ensemble:         Low (simple averaging)")
print(f"  Stacking App1:    Low (LogReg on 2 outputs)")
print(f"  Stacking App2:    High (LightGBM on inputs + 2 outputs)")


COMPARISON n0: Ensemble vs 2 Stacking Approaches

                         Model       F1      AUC  Accuracy  Precision   Recall
            Ensemble (5a+5b)/2 0.916227 0.949015  0.917722   0.915378 0.917722
  Stacking Approach 1 (LogReg) 0.908599 0.950303  0.909283   0.908027 0.909283
Stacking Approach 2 (LightGBM) 0.910130 0.941696  0.915612   0.911999 0.915612

🏆 Best n0: Ensemble (5a+5b)/2 (F1: 0.9162)

COMPLEXITY:
  Ensemble:         Low (simple averaging)
  Stacking App1:    Low (LogReg on 2 outputs)
  Stacking App2:    High (LightGBM on inputs + 2 outputs)


# MODEL 5 n1: LightGBM + CatBoost

## 11. Build Model 5 n1 Sub-models (5a + 5b)

In [21]:
print("\n" + "="*80)
print("MODEL 5 n1: LightGBM (5a) + CatBoost (5b)")
print("="*80)

# Sub-model 5a: LightGBM
print("\nTraining 5a: LightGBM...")
model5_n1_5a = LGBMClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=7, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, is_unbalance=True,
    random_state=42, verbose=-1
)
model5_n1_5a.fit(X_train_scaled, y_train)
print("✅ Model 5a trained")

# Sub-model 5b: CatBoost
print("\nTraining 5b: CatBoost...")
model5_n1_5b = CatBoostClassifier(
    iterations=200, learning_rate=0.05, depth=7,
    auto_class_weights='Balanced', random_state=42, verbose=0
)
model5_n1_5b.fit(X_train_scaled, y_train)
print("✅ Model 5b trained")

# Get predictions for validation
print("\nGenerating predictions...")
y_val_pred_5a_n1 = model5_n1_5a.predict(X_val_scaled)
y_val_proba_5a_n1 = model5_n1_5a.predict_proba(X_val_scaled)[:, 1]

y_val_pred_5b_n1 = model5_n1_5b.predict(X_val_scaled)
y_val_proba_5b_n1 = model5_n1_5b.predict_proba(X_val_scaled)[:, 1]

print("✅ Predictions generated")


MODEL 5 n1: LightGBM (5a) + CatBoost (5b)

Training 5a: LightGBM...
✅ Model 5a trained

Training 5b: CatBoost...
✅ Model 5b trained

Generating predictions...
✅ Predictions generated


## 12. Stacking Approach 1 n1: LogReg on Outputs Only

In [23]:
print("\n" + "="*80)
print("APPROACH 1 n1: LogReg Meta-learner on 2 Outputs")
print("="*80)

# Get train predictions for meta-features
y_train_proba_5a_n1 = model5_n1_5a.predict_proba(X_train_scaled)[:, 1]
y_train_proba_5b_n1 = model5_n1_5b.predict_proba(X_train_scaled)[:, 1]

# Create meta-features (outputs only)
X_train_meta_n1_app1 = np.column_stack([y_train_proba_5a_n1, y_train_proba_5b_n1])
X_val_meta_n1_app1 = np.column_stack([y_val_proba_5a_n1, y_val_proba_5b_n1])

print(f"Meta-features shape: Train {X_train_meta_n1_app1.shape} | Val {X_val_meta_n1_app1.shape}")

# Train meta-learner (LogReg)
print("\nTraining LogReg meta-learner...")
stacking_n1_app1 = LogisticRegression(
    penalty='l2', C=1.0, max_iter=2000, solver='lbfgs',
    class_weight='balanced', random_state=42, verbose=0
)
stacking_n1_app1.fit(X_train_meta_n1_app1, y_train)
print("✅ Meta-learner trained")

# Predictions
y_val_pred_stack_n1_app1 = stacking_n1_app1.predict(X_val_meta_n1_app1)
y_val_proba_stack_n1_app1 = stacking_n1_app1.predict_proba(X_val_meta_n1_app1)[:, 1]

# Metrics
f1_stack_n1_app1 = f1_score(y_val, y_val_pred_stack_n1_app1, average='weighted')
auc_stack_n1_app1 = roc_auc_score(y_val, y_val_proba_stack_n1_app1)
acc_stack_n1_app1 = accuracy_score(y_val, y_val_pred_stack_n1_app1)
prec_stack_n1_app1 = precision_score(y_val, y_val_pred_stack_n1_app1, average='weighted')
rec_stack_n1_app1 = recall_score(y_val, y_val_pred_stack_n1_app1, average='weighted')

print(f"\n✅ Approach 1 Results:")
print(f"   F1: {f1_stack_n1_app1:.4f} | AUC: {auc_stack_n1_app1:.4f} | Acc: {acc_stack_n1_app1:.4f}")
print(f"   Prec: {prec_stack_n1_app1:.4f} | Rec: {rec_stack_n1_app1:.4f}")


APPROACH 1 n1: LogReg Meta-learner on 2 Outputs
Meta-features shape: Train (3788, 2) | Val (474, 2)

Training LogReg meta-learner...
✅ Meta-learner trained

✅ Approach 1 Results:
   F1: 0.9171 | AUC: 0.9409 | Acc: 0.9177
   Prec: 0.9166 | Rec: 0.9177


## 13. Stacking Approach 2 n1: LightGBM on Inputs + Outputs

In [24]:
print("\n" + "="*80)
print("APPROACH 2 n1: LightGBM Meta-learner on Input Features + 2 Outputs")
print("="*80)

# Create meta-features (inputs + outputs)
X_train_meta_n1_app2 = np.column_stack([
    y_train_proba_5a_n1, y_train_proba_5b_n1,
    X_train_scaled
])
X_val_meta_n1_app2 = np.column_stack([
    y_val_proba_5a_n1, y_val_proba_5b_n1,
    X_val_scaled
])

print(f"Meta-features shape: Train {X_train_meta_n1_app2.shape} | Val {X_val_meta_n1_app2.shape}")

# Train meta-learner (LightGBM)
print("\nTraining LightGBM meta-learner...")
stacking_n1_app2 = LGBMClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=7, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, is_unbalance=True,
    random_state=42, verbose=-1
)
stacking_n1_app2.fit(X_train_meta_n1_app2, y_train)
print("✅ Meta-learner trained")

# Predictions
y_val_pred_stack_n1_app2 = stacking_n1_app2.predict(X_val_meta_n1_app2)
y_val_proba_stack_n1_app2 = stacking_n1_app2.predict_proba(X_val_meta_n1_app2)[:, 1]

# Metrics
f1_stack_n1_app2 = f1_score(y_val, y_val_pred_stack_n1_app2, average='weighted')
auc_stack_n1_app2 = roc_auc_score(y_val, y_val_proba_stack_n1_app2)
acc_stack_n1_app2 = accuracy_score(y_val, y_val_pred_stack_n1_app2)
prec_stack_n1_app2 = precision_score(y_val, y_val_pred_stack_n1_app2, average='weighted')
rec_stack_n1_app2 = recall_score(y_val, y_val_pred_stack_n1_app2, average='weighted')

print(f"\n✅ Approach 2 Results:")
print(f"   F1: {f1_stack_n1_app2:.4f} | AUC: {auc_stack_n1_app2:.4f} | Acc: {acc_stack_n1_app2:.4f}")
print(f"   Prec: {prec_stack_n1_app2:.4f} | Rec: {rec_stack_n1_app2:.4f}")


APPROACH 2 n1: LightGBM Meta-learner on Input Features + 2 Outputs
Meta-features shape: Train (3788, 1693) | Val (474, 1693)

Training LightGBM meta-learner...
✅ Meta-learner trained

✅ Approach 2 Results:
   F1: 0.8097 | AUC: 0.9152 | Acc: 0.8523
   Prec: 0.8483 | Rec: 0.8523


## 14. Comparison n1: Ensemble vs 2 Stacking

In [25]:
print("\n" + "="*80)
print("COMPARISON n1: Ensemble vs 2 Stacking Approaches")
print("="*80)

# Calculate Ensemble (average of 5a + 5b)
y_val_proba_ensemble_n1 = (y_val_proba_5a_n1 + y_val_proba_5b_n1) / 2
y_val_pred_ensemble_n1 = (y_val_proba_ensemble_n1 >= 0.5).astype(int)

# Metrics for Ensemble
f1_ensemble_n1 = f1_score(y_val, y_val_pred_ensemble_n1, average='weighted')
auc_ensemble_n1 = roc_auc_score(y_val, y_val_proba_ensemble_n1)
acc_ensemble_n1 = accuracy_score(y_val, y_val_pred_ensemble_n1)
prec_ensemble_n1 = precision_score(y_val, y_val_pred_ensemble_n1, average='weighted')
rec_ensemble_n1 = recall_score(y_val, y_val_pred_ensemble_n1, average='weighted')

results_n1 = pd.DataFrame({
    'Model': ['Ensemble (5a+5b)/2', 'Stacking Approach 1 (LogReg)', 'Stacking Approach 2 (LightGBM)'],
    'F1': [f1_ensemble_n1, f1_stack_n1_app1, f1_stack_n1_app2],
    'AUC': [auc_ensemble_n1, auc_stack_n1_app1, auc_stack_n1_app2],
    'Accuracy': [acc_ensemble_n1, acc_stack_n1_app1, acc_stack_n1_app2],
    'Precision': [prec_ensemble_n1, prec_stack_n1_app1, prec_stack_n1_app2],
    'Recall': [rec_ensemble_n1, rec_stack_n1_app1, rec_stack_n1_app2]
})

print("\n" + results_n1.to_string(index=False))
best_idx_n1 = results_n1['F1'].idxmax()
print(f"\n🏆 Best n1: {results_n1.loc[best_idx_n1, 'Model']} (F1: {results_n1.loc[best_idx_n1, 'F1']:.4f})")

# Complexity
print(f"\nCOMPLEXITY:")
print(f"  Ensemble:         Low (simple averaging)")
print(f"  Stacking App1:    Low (LogReg on 2 outputs)")
print(f"  Stacking App2:    High (LightGBM on inputs + 2 outputs)")


COMPARISON n1: Ensemble vs 2 Stacking Approaches

                         Model       F1      AUC  Accuracy  Precision   Recall
            Ensemble (5a+5b)/2 0.917518 0.941099  0.917722   0.917328 0.917722
  Stacking Approach 1 (LogReg) 0.917102 0.940910  0.917722   0.916597 0.917722
Stacking Approach 2 (LightGBM) 0.809659 0.915182  0.852321   0.848289 0.852321

🏆 Best n1: Ensemble (5a+5b)/2 (F1: 0.9175)

COMPLEXITY:
  Ensemble:         Low (simple averaging)
  Stacking App1:    Low (LogReg on 2 outputs)
  Stacking App2:    High (LightGBM on inputs + 2 outputs)


## 15. FINAL COMPARISON: All 6 Models

In [26]:
print("\n" + "="*80)
print("FINAL COMPARISON: All 6 Models (3 from n0 + 3 from n1)")
print("="*80)

final_results = pd.DataFrame({
    'Model': [
        'n0-Ensemble (5a+5b)/2',
        'n0-Stacking App1 (LogReg)',
        'n0-Stacking App2 (LightGBM)',
        'n1-Ensemble (5a+5b)/2',
        'n1-Stacking App1 (LogReg)',
        'n1-Stacking App2 (LightGBM)'
    ],
    'F1': [f1_ensemble_n0, f1_stack_n0_app1, f1_stack_n0_app2, f1_ensemble_n1, f1_stack_n1_app1, f1_stack_n1_app2],
    'AUC': [auc_ensemble_n0, auc_stack_n0_app1, auc_stack_n0_app2, auc_ensemble_n1, auc_stack_n1_app1, auc_stack_n1_app2],
    'Accuracy': [acc_ensemble_n0, acc_stack_n0_app1, acc_stack_n0_app2, acc_ensemble_n1, acc_stack_n1_app1, acc_stack_n1_app2],
    'Precision': [prec_ensemble_n0, prec_stack_n0_app1, prec_stack_n0_app2, prec_ensemble_n1, prec_stack_n1_app1, prec_stack_n1_app2],
    'Recall': [rec_ensemble_n0, rec_stack_n0_app1, rec_stack_n0_app2, rec_ensemble_n1, rec_stack_n1_app1, rec_stack_n1_app2]
})

print("\n" + final_results.to_string(index=False))

best_idx = final_results['F1'].idxmax()
print(f"\n🏆 BEST MODEL OVERALL")
print(f"Model: {final_results.loc[best_idx, 'Model']}")
print(f"F1: {final_results.loc[best_idx, 'F1']:.4f} | AUC: {final_results.loc[best_idx, 'AUC']:.4f}")

print(f"\n" + "="*80)
print("COMPLEXITY ANALYSIS (Like Section 11 in n0 & n1)")
print("="*80)

complexity = pd.DataFrame({
    'Model': final_results['Model'],
    'Complexity': [
        'Low (simple averaging)',
        'Low (LogReg on 2 outputs)',
        'High (LightGBM on inputs + 2 outputs)',
        'Low (simple averaging)',
        'Low (LogReg on 2 outputs)',
        'High (LightGBM on inputs + 2 outputs)'
    ],
    'Training Time': ['Fast', 'Fast', 'Slow', 'Fast', 'Fast', 'Slow'],
    'Interpretability': ['High', 'Medium', 'Low', 'High', 'Medium', 'Low']
})

print("\n" + complexity.to_string(index=False))

print(f"\n✅ Analysis Complete!")


FINAL COMPARISON: All 6 Models (3 from n0 + 3 from n1)

                      Model       F1      AUC  Accuracy  Precision   Recall
      n0-Ensemble (5a+5b)/2 0.916227 0.949015  0.917722   0.915378 0.917722
  n0-Stacking App1 (LogReg) 0.908599 0.950303  0.909283   0.908027 0.909283
n0-Stacking App2 (LightGBM) 0.910130 0.941696  0.915612   0.911999 0.915612
      n1-Ensemble (5a+5b)/2 0.917518 0.941099  0.917722   0.917328 0.917722
  n1-Stacking App1 (LogReg) 0.917102 0.940910  0.917722   0.916597 0.917722
n1-Stacking App2 (LightGBM) 0.809659 0.915182  0.852321   0.848289 0.852321

🏆 BEST MODEL OVERALL
Model: n1-Ensemble (5a+5b)/2
F1: 0.9175 | AUC: 0.9411

COMPLEXITY ANALYSIS (Like Section 11 in n0 & n1)

                      Model                            Complexity Training Time Interpretability
      n0-Ensemble (5a+5b)/2                Low (simple averaging)          Fast             High
  n0-Stacking App1 (LogReg)             Low (LogReg on 2 outputs)          Fast           

## 16. Model Complexity & Computational Analysis

In [30]:
import time
import pickle
import numpy as np

print("\n" + "="*80)
print("MODEL COMPLEXITY & COMPUTATIONAL ANALYSIS (+ FLOPs)")
print("="*80)

# List of models with their types and input dimensions
models_info = {
    'n0-Ensemble': {
        'type': 'Ensemble (Average)',
        'input_dim': 2,  # 2 base model predictions
        'n_base_models': 2,
        'description': 'Simple averaging of 2 sub-models'
    },
    'n0-Stacking App1': {
        'model': stacking_n0_app1,
        'type': 'LogReg',
        'input_dim': X_val_meta_n0_app1.shape[1],
        'description': 'LogReg on 2 outputs'
    },
    'n0-Stacking App2': {
        'model': stacking_n0_app2,
        'type': 'LightGBM',
        'input_dim': X_val_meta_n0_app2.shape[1],
        'description': 'LightGBM on inputs + 2 outputs'
    },
    'n1-Ensemble': {
        'type': 'Ensemble (Average)',
        'input_dim': 2,  # 2 base model predictions
        'n_base_models': 2,
        'description': 'Simple averaging of 2 sub-models'
    },
    'n1-Stacking App1': {
        'model': stacking_n1_app1,
        'type': 'LogReg',
        'input_dim': X_val_meta_n1_app1.shape[1],
        'description': 'LogReg on 2 outputs'
    },
    'n1-Stacking App2': {
        'model': stacking_n1_app2,
        'type': 'LightGBM',
        'input_dim': X_val_meta_n1_app2.shape[1],
        'description': 'LightGBM on inputs + 2 outputs'
    },
}

# Calculate model complexity
complexity_data = []

for name, info in models_info.items():
    model_type = info['type']
    input_dim = info['input_dim']
    description = info['description']
    
    # Get number of parameters and FLOPs based on model type
    if model_type == 'Ensemble (Average)':
        # Ensemble: just averaging 2 predictions
        n_params = 0  # No learnable parameters
        flops_per_sample = 2  # 2 additions for averaging
        inference_time_ms = 0.001  # Negligible
        memory_kb = 0.1
    elif model_type == 'LogReg':
        # LogisticRegression: coef (input_dim x n_classes) + intercept
        model = info['model']
        n_params = model.coef_.shape[0] * model.coef_.shape[1] + model.intercept_.shape[0]
        # FLOPs: input_dim multiplies + input_dim adds
        flops_per_sample = 2 * input_dim
        
        # Measure inference time (average of 10 runs)
        times = []
        for _ in range(10):
            start_time = time.perf_counter()
            _ = model.predict(np.random.randn(100, input_dim))
            times.append((time.perf_counter() - start_time) * 1000)
        inference_time_ms = np.mean(times) / 100  # per sample
        
        # Model memory
        try:
            model_bytes = pickle.dumps(model)
            memory_kb = len(model_bytes) / 1024
        except:
            memory_kb = 0
            
    else:  # LightGBM
        model = info['model']
        n_estimators = model.n_estimators
        # Estimate parameters: roughly n_estimators * leaves per tree
        n_params = n_estimators * 31 * input_dim  # 31 leaves default
        # FLOPs: tree traversal = n_estimators * avg_depth
        flops_per_sample = n_estimators * 7 * 2  # depth=7, *2 for comparison+ops
        
        # Measure inference time (average of 10 runs)
        times = []
        for _ in range(10):
            start_time = time.perf_counter()
            _ = model.predict(np.random.randn(100, input_dim))
            times.append((time.perf_counter() - start_time) * 1000)
        inference_time_ms = np.mean(times) / 100  # per sample
        
        # Model memory
        try:
            model_bytes = pickle.dumps(model)
            memory_kb = len(model_bytes) / 1024
        except:
            memory_kb = 0
    
    complexity_data.append({
        'Model': name,
        'Type': model_type,
        'Input Dims': input_dim,
        'Parameters': f"{n_params:,}" if n_params > 0 else "0",
        'FLOPs/Sample': int(flops_per_sample) if flops_per_sample > 0 else "0",
        'Inference (ms)': round(inference_time_ms, 4),
        'Memory (KB)': round(memory_kb, 2),
        'Description': description
    })

# Create DataFrame
complexity_df = pd.DataFrame(complexity_data)

print("\n" + "─"*160)
print("STACKING MODELS - COMPLEXITY METRICS (Including FLOPs Estimation)")
print("─"*160 + "\n")
print(complexity_df.to_string(index=False))

print(f"\n" + "="*80)
print("EFFICIENCY ANALYSIS: F1 Score vs Model Complexity")
print("="*80)

# Efficiency metrics
efficiency_data = []
for idx, row in final_results.iterrows():
    model_name = row['Model']
    f1_score = row['F1']
    
    # Get complexity info
    complexity_row = next((c for c in complexity_data if model_name in c['Model']), None)
    if complexity_row:
        try:
            params = int(complexity_row['Parameters'].replace(',', ''))
        except:
            params = 1  # avoid division by zero for ensemble
        
        flops = int(complexity_row['FLOPs/Sample']) if complexity_row['FLOPs/Sample'] != '0' else 1
        
        efficiency_data.append({
            'Model': model_name,
            'F1 Score': f"{f1_score:.4f}",
            'Parameters': complexity_row['Parameters'],
            'F1/Parameters': f"{f1_score/max(params, 1):.6f}" if params > 0 else "N/A",
            'FLOPs/Sample': complexity_row['FLOPs/Sample'],
            'F1/FLOPs': f"{f1_score/max(flops, 1):.6f}" if flops > 0 else "N/A",
            'Inference (ms)': complexity_row['Inference (ms)']
        })

efficiency_df = pd.DataFrame(efficiency_data)
print("\n" + efficiency_df.to_string(index=False))

print(f"\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)
print(f"\n✅ Most Efficient Models (Low Complexity, High F1):")
print(f"   • Ensemble approaches: Zero learnable parameters, negligible FLOPs")
print(f"   • Stacking App1 (LogReg): Linear model complexity")
print(f"\n⚠️  Complex Models (High Complexity, Variable F1):")
print(f"   • Stacking App2 (LightGBM): Tree-based with higher FLOPs")
print(f"\n💡 Trade-off Analysis:")
print(f"   • Ensemble: Simplest (O(1) complexity), competitive F1 scores")
print(f"   • Stacking App1: Low complexity with meta-learner, slightly lower F1")
print(f"   • Stacking App2: Higher complexity, mixed results (better n0, worse n1)")

print(f"\n✅ Analysis Complete!")


MODEL COMPLEXITY & COMPUTATIONAL ANALYSIS (+ FLOPs)

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
STACKING MODELS - COMPLEXITY METRICS (Including FLOPs Estimation)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

           Model               Type  Input Dims Parameters  FLOPs/Sample  Inference (ms)  Memory (KB)                      Description
     n0-Ensemble Ensemble (Average)           2          0             2          0.0010         0.10 Simple averaging of 2 sub-models
n0-Stacking App1             LogReg           2          3             4          0.0013         0.72              LogReg on 2 outputs
n0-Stacking App2           LightGBM        1693 10,496,600          2800          0.0449       635.89   LightGBM on inputs + 2 outputs
     n1-Ensemble 